In [1]:
# ============================================================
# TEST — 1. LOAD THE ORIGINAL TR CATALOG
# ============================================================

from pathlib import Path
import pandas as pd
from google.colab import files

raw_files = list(Path("/content").glob("TR_CATALOG_RAW*"))

if len(raw_files) != 1:
    print("Please upload exactly one TR_CATALOG_RAW.tsv file.")
    uploaded = files.upload()

    raw_files = [
        Path("/content") / name
        for name in uploaded
        if Path(name).name.startswith("TR_CATALOG_RAW")
    ]

if len(raw_files) != 1:
    raise FileNotFoundError(
        "Exactly one TR_CATALOG_RAW.tsv file is required."
    )

RAW = pd.read_csv(
    raw_files[0],
    sep="\t",
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False
)

required_columns = [
    "Chromosome",
    "Start",
    "End",
    "Period",
    "Copy_Number",
    "Consensus",
    "Repeat_Sequence"
]

missing_columns = [
    column for column in required_columns
    if column not in RAW.columns
]

if missing_columns:
    raise ValueError(
        f"Missing RAW columns: {missing_columns}"
    )

for column in ["Start", "End", "Period"]:
    RAW[column] = pd.to_numeric(
        RAW[column],
        errors="coerce"
    )

print("RAW catalog loaded.")
print("Rows:", len(RAW))
print("Columns:", len(RAW.columns))

Please upload exactly one TR_CATALOG_RAW.tsv file.


Saving TR_CATALOG_RAW.tsv to TR_CATALOG_RAW.tsv
RAW catalog loaded.
Rows: 39098
Columns: 16


In [2]:
# ============================================================
# TEST — 2. DEFINE THE INITIAL CANDIDATE SET
# ============================================================

candidate_specs = [
    ("AB_1745", "AB", "I", 1919838, 1922036, 35),
    ("AB_32622", "AB", "V", 18169382, 18171317, 31),
    ("AB_32946", "AB", "V", 18567606, 18568227, 10),
    ("AB_33142", "AB", "V", 18694160, 18694786, 10),

    ("MS_463", "MS", "I", 766872, 767550, 22),
    ("MS_6476", "MS", "I", 14300049, 14301120, 35),
    ("MS_6493", "MS", "I", 14303513, 14304584, 35),
    ("MS_33627", "MS", "V", 19122509, 19124517, 66),
    ("MS_29900", "MS", "V", 6176204, 6183407, 234),

    ("C_1130", "C", "I", 1391387, 1391533, 10),
    ("C_12721", "C", "II", 13785721, 13785881, 11),
    ("C_12801", "C", "II", 13864621, 13864779, 11),
    ("C_15368", "C", "III", 1698655, 1698809, 11),
    ("C_10956", "C", "II", 12197271, 12197445, 12),

    ("D_18513", "D", "III", 11390291, 11390333, 7),
    ("D_32101", "D", "V", 17361885, 17361931, 7),

    ("E_10203", "E", "II", 8833974, 8834023, 8),
    ("E_19317", "E", "III", 12070120, 12070169, 8),

    ("AB_APO_X_4031472_4056451",
     "AB", "X", 4031472, 4056451, None),

    ("AB_IV_6682656_6731929",
     "AB", "IV", 6682656, 6731929, 174),

    ("AB_III_13584282_13599363",
     "AB", "III", 13584282, 13599363, 51),

    ("AB_III_13584328_13599377",
     "AB", "III", 13584328, 13599377, 16),

    ("AB_IV_1902960_1903281",
     "AB", "IV", 1902960, 1903281, 6),

    ("MS_III_1239493_1240806",
     "MS", "III", 1239493, 1240806, 35),

    ("C_I_355869_356717",
     "C", "I", 355869, 356717, 19),

    ("DE_IV_6682656_6704725",
     "D/E", "IV", 6682656, 6704725, 172)
]

assert len(candidate_specs) == 26

print("Initial candidate set:", len(candidate_specs))

Initial candidate set: 26


In [9]:
# ============================================================
# TEST — 3. RETRIEVE AND VALIDATE RAW SEQUENCES
# ============================================================

candidate_records = []

for tr_id, lineage, chromosome, start, end, expected_period in candidate_specs:

    # Initially filter by genomic coordinates
    matches = RAW[
        (RAW["Chromosome"].str.upper() == chromosome.upper()) &
        (RAW["Start"] == start) &
        (RAW["End"] == end)
    ]

    # If there are multiple matches, use the expected_period to resolve ambiguity
    if len(matches) > 1 and expected_period is not None:
        matches = matches[matches["Period"] == expected_period]

    if len(matches) != 1:
        raise ValueError(
            f"{tr_id}: expected one RAW match, found {len(matches)}."
        )

    record = matches.iloc[0]

    period = int(record["Period"])

    if expected_period is not None and period != expected_period:
        raise ValueError(
            f"{tr_id}: expected period {expected_period}, "
            f"found {period}."
        )

    sequence = str(record["Repeat_Sequence"]).upper()

    invalid_bases = sorted(set(sequence) - set("ACGT"))

    if invalid_bases:
        raise ValueError(
            f"{tr_id}: invalid DNA symbols {invalid_bases}."
        )

    complete_count, trailing_nt = divmod(
        len(sequence),
        period
    )

    complete_sequence = sequence[
        :complete_count * period
    ]

    monomers = [
        complete_sequence[i:i + period]
        for i in range(0, len(complete_sequence), period)
    ]

    reference_monomer = str(
        record["Consensus"]
    ).upper()

    if set(reference_monomer) - set("ACGT"):
        raise ValueError(
            f"{tr_id}: invalid reference monomer."
        )

    # We tolerate small length discrepancies between Consensus and Period,
    # simply printing a warning instead of blocking execution.
    if len(reference_monomer) != period:
        print(
            f"[WARNING] {tr_id}: reference monomer length ({len(reference_monomer)}) "
            f"differs from period ({period})."
        )

    distinct_monomers = len(set(monomers))

    candidate_records.append({
        "Lineage": lineage,
        "TR_ID": tr_id,
        "Chromosome": chromosome,
        "Start": start,
        "End": end,
        "Period_nt": period,
        "Sequence_length_nt": len(sequence),
        "Complete_monomers": complete_count,
        "Reference_monomer": reference_monomer,
        "Monomer_length_nt": len(reference_monomer),
        "Trailing_nt": trailing_nt,
        "Distinct_monomers": distinct_monomers,
        "All_monomers_distinct": (
            distinct_monomers == complete_count
        ),
        "Selection_status": "Initial candidate"
    })

df_candidates = pd.DataFrame(candidate_records)

print("RAW retrieval completed.")
print("Validated candidates:", len(df_candidates))
print(
    "Candidates with all distinct monomers:",
    df_candidates["All_monomers_distinct"].sum()
)

display(df_candidates)

[WARNING] MS_463: reference monomer length (21) differs from period (22).
[WARNING] C_12801: reference monomer length (10) differs from period (11).
[WARNING] C_10956: reference monomer length (11) differs from period (12).
[WARNING] E_19317: reference monomer length (9) differs from period (8).
[WARNING] AB_III_13584282_13599363: reference monomer length (48) differs from period (51).
[WARNING] AB_III_13584328_13599377: reference monomer length (17) differs from period (16).
[WARNING] MS_III_1239493_1240806: reference monomer length (34) differs from period (35).
[WARNING] C_I_355869_356717: reference monomer length (20) differs from period (19).
RAW retrieval completed.
Validated candidates: 26
Candidates with all distinct monomers: 15


,Lineage,TR_ID,Chromosome,Start,End,Period_nt,Sequence_length_nt,Complete_monomers,Reference_monomer,Monomer_length_nt,Trailing_nt,Distinct_monomers,All_monomers_distinct,Selection_status
0,AB,AB_1745,I,1919838,1922036,35,2199,62,TTGCCGATTTGCCGGAAATTTTCATTTTTGGCAAA,35,29,52,False,Initial candidate
1,AB,AB_32622,V,18169382,18171317,31,1936,62,AAATTCAAATTTTCAGTCAATTTTTGGCGGG,31,14,48,False,Initial candidate
2,AB,AB_32946,V,18567606,18568227,10,622,62,AAAATTCCAG,10,2,50,False,Initial candidate
3,AB,AB_33142,V,18694160,18694786,10,627,62,ATTCCAGAAA,10,7,52,False,Initial candidate
4,MS,MS_463,I,766872,767550,22,679,30,TCCCTACAGTACCCTACAGTA,21,19,30,True,Initial candidate
5,MS,MS_6476,I,14300049,14301120,35,1072,30,AATTTTCAATTCCGGCAATTTGCCGATTTGCCGGA,35,22,30,True,Initial candidate
6,MS,MS_6493,I,14303513,14304584,35,1072,30,AATTTTCAATTCCGGCAATTTGCCGATTTGCCGGA,35,22,30,True,Initial candidate
7,MS,MS_33627,V,19122509,19124517,66,2009,30,TTAGCAGCGCCCAAAATCTTGTAGTTTGTAGTCTTAGCAGCGCCCA...,66,29,30,True,Initial candidate
8,MS,MS_29900,V,6176204,6183407,234,7204,30,CAGATGTTTCTTTTGATTCCACAGGTGCAAGTTTCTCAACTGTTGG...,234,184,30,True,Initial candidate
9,C,C_1130,I,1391387,1391533,10,147,14,TTTTTATAGA,10,7,14,True,Initial candidate


In [8]:
# ============================================================
# TEST — 4. SAVE THE INITIAL CANDIDATE TABLE
# ============================================================

output_file = "/content/TEST_initial_26_candidates.csv"

df_candidates.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

files.download(output_file)

Saved: /content/TEST_initial_26_candidates.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>